In [1]:
import os
import random
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from human_vae_model import PointNetVAE

# モデルを作成してロード
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointNetVAE(latent_dim=64, num_points=6890).to(device)
model.load_state_dict(torch.load("pointnet_vae.pth", map_location=device))
model.eval()

input = x = torch.rand(6890, 3).unsqueeze(0) # (1, 6890, 3)
input = input.to(device)
                
# 推論
with torch.no_grad():
    output, mu, logvar = model(input)

# バッチ次元を除く → (6890, 3)
points = output.cpu()[0]

# 3D表示
fig = go.Figure(data=[go.Scatter3d(
    x=points[:, 0],
    y=points[:, 1],
    z=points[:, 2],
    mode='markers',
    marker=dict(size=2, color='blue')
)])

fig.update_layout(
    title='3D Point Cloud',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'  # これで軸の倍率が等しくなる
    ),
    margin=dict(l=0, r=0, b=0, t=30)
)

fig.show()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
